In [1]:
import pandas as pd
import sys 
from tqdm import tqdm
from rerankers import Reranker
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
sys.path.append("../../utils")
from cs_pipeline import GraphCS, apply_cs_component
from reranking import (apply_reranking_component)
from processing import (json_file_access,
                              evaluate)
import networkit as nk
import networkx as nx

import warnings
warnings.filterwarnings("ignore")

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
configuration = {
    "KG": "kg2",
    "cs_method": "LFMLocal",
    "initial_node_count": 10,
    "params": {"alpha":1.0, "Q": "M"}
}

node_df = pd.read_parquet("../../03_data/preprocessed_data/nodes_beir_dbpedia.parquet")

if configuration["KG"] == "kg1":
    edges = pd.read_parquet('../../03_data/preprocessed_data/pagelink_edges_beir_dbpedia.parquet')
elif configuration["KG"] == "kg2":
    edges = pd.read_parquet('../../03_data/preprocessed_data/wikilink_edges_beir_dbpedia.parquet')
else: 
    raise ValueError("KG not found")

# Load the results from IR-Component
results_rq1 = json_file_access("../../05_results/rq1/results_rq1.json", "r")
# Initiate CS-Component
Graph = GraphCS(node_df, edges)

ranker = Reranker('mixedbread-ai/mxbai-rerank-base-v1', 
                  model_type='cross-encoder')

evaluator = EvaluateRetrieval()

corpus, queries, qrels = GenericDataLoader(data_folder="../../03_data/raw_data/beir_dbpedia/dbpedia-entity").load(split="test")

Building Graph from given Data...
Graph built
Loading TransformerRanker model mixedbread-ai/mxbai-rerank-base-v1 (this message can be suppressed by setting verbose=0)
No device set
Using device cpu
No dtype set
Using dtype torch.float32
Loaded model mixedbread-ai/mxbai-rerank-base-v1
Using device cpu.
Using dtype torch.float32.


100%|██████████| 4635922/4635922 [00:50<00:00, 91674.30it/s] 


## Rearanking on top of RQ1

In [ ]:
# Apply Reranking to the root results
def convert_dict_of_dict_of_dict_to_dict_of_dict_of_list(input_dict):
    output_dict = {}
    for key1, sub_dict1 in input_dict.items():
        output_dict[key1] = {}
        for key2, sub_dict2 in sub_dict1.items():
            output_dict[key1][key2] = list(sub_dict2.keys())
    return output_dict

cs_method  = "None"
results_reranked ={"boolean":{},
          "bm25":{},
          "vector":{},
          "hybrid":{}}

for search_type in results_reranked.keys():
    print("Processing search type:", search_type)
    results_reranked[search_type] = apply_reranking_component(convert_dict_of_dict_of_dict_to_dict_of_dict_of_list(results_rq1)[search_type], ranker, queries, node_df)
json_file_access(f"../../05_results/rq2/reranking/results_rq2_{cs_method}_reranked.json", "w", results_reranked)

evaluation = evaluate(results_reranked, [10, 100], evaluator, qrels)
evaluation.to_csv(f"../../05_results/rq2/evaluation/evaluation_rq2_{cs_method}_reranked.csv", index=False)
evaluation.round(3)

Processing search type: boolean


  0%|          | 2/400 [00:20<1:04:38,  9.74s/it]

INEX_LD-2009039


  1%|          | 4/400 [00:46<1:20:03, 12.13s/it]

INEX_LD-2009061


...

INEX_LD-20120111
INEX_LD-20120121


  6%|▌         | 22/400 [01:47<02:48,  2.25it/s]

INEX_LD-20120122
INEX_LD-20120131


  6%|▌         | 24/400 [01:48<01:51,  3.37it/s]

INEX_LD-20120211
INEX_LD-20120221


## Experiments with Community Search:

In [ ]:
# Execute CS-Component
results ={"boolean": {},
          "bm25":{},
          "vector":{}, 
          "hybrid":{}}
results_all = {"boolean": {},
          "bm25":{},
          "vector":{}, 
          "hybrid":{}}


for search_type in results.keys():
    print("Processing search type:", search_type)
    results[search_type], results_all[search_type] = apply_cs_component(results_rq1[search_type], Graph, configuration["cs_method"], configuration["initial_node_count"], **configuration["params"])
json_file_access(f"../../05_results/rq2/cs/{configuration['KG']}_results_rq2_cs_{configuration['cs_method']}_{configuration['initial_node_count']}.json", "w", results)
json_file_access(f"../../05_results/rq2/cs/{configuration['KG']}_results_rq2_cs_{configuration['cs_method']}_{configuration['initial_node_count']}_all_community.json", "w", results_all)

Processing search type: boolean


  2%|▏         | 7/400 [00:00<00:22, 17.12it/s]

In [ ]:
# Evaluate CS-Component
results = json_file_access(f"../../05_results/rq2/cs/{configuration['KG']}_results_rq2_cs_{configuration['cs_method']}_{configuration['initial_node_count']}.json", "r")

results_reranked ={"boolean":{},
          "bm25":{},
          "vector":{},
          "hybrid":{}}

for search_type in results.keys():
    print("Processing search type:", search_type)
    results_reranked[search_type] = apply_reranking_component(results[search_type], ranker, queries, node_df)
json_file_access(f"../../05_results/rq2/reranking/{configuration['KG']}_results_rq2_{configuration['cs_method']}_{configuration['initial_node_count']}_reranked.json", "w", results_reranked)

evaluation = evaluate(results_reranked, [10, 100], evaluator, qrels)
evaluation.to_csv(f"../../05_results/rq2/evaluation/{configuration['KG']}_results_rq2_{configuration['cs_method']}_{configuration['initial_node_count']}_reranked.csv", index=False)
evaluation.round(3)